In [ ]:
import os
import requests
from dotenv import load_dotenv
from typing import Annotated
from typing_extensions import TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

# Import LangGraph components
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv()
openrouter_key = os.getenv("Dots3_API")

# 1. The Tool (Same as before)
@tool
def get_player_price_and_form(player_name: str) -> str:
    """Fetches the current price and form of a Premier League player from the official FPL API."""
    url = "https://fantasy.premierleague.com/api/bootstrap-static/"
    response = requests.get(url)
    if response.status_code != 200:
        return "Failed to fetch data."
    data = response.json()
    for player in data['elements']:
        full_name = f"{player['first_name']} {player['second_name']}".lower()
        if player_name.lower() in full_name:
            price = player['now_cost'] / 10.0 
            form = player['form']
            return f"{player['first_name']} {player['second_name']} costs £{price}m and has a form rating of {form}."
    return f"Could not find player {player_name}."

tools = [get_player_price_and_form]

# 2. The LLM (Brain)
llm = ChatOpenAI(
    model="dots-studio/dots-3-note-preview:free",
    openai_api_key="Dots3_API",
    openai_api_base="https://openrouter.ai/api/v1"
)
# We must explicitly "bind" the tools so the LLM knows they exist
llm_with_tools = llm.bind_tools(tools)

# 3. Define the State (Memory)
# This dictates that our graph's memory is a list of messages. 
# `add_messages` ensures new messages are appended, not overwritten.
class State(TypedDict):
    messages: Annotated[list, add_messages]

# 4. Define the LLM Node Function
def chatbot_node(state: State):
    """This node passes the current memory to the LLM and returns its response."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# 5. Build the LangGraph Flowchart
graph_builder = StateGraph(State)

# Add our two nodes
graph_builder.add_node("chatbot", chatbot_node)
graph_builder.add_node("tools", ToolNode(tools)) # Pre-built node that runs Python tools

# Draw the edges (the paths between nodes)
graph_builder.add_edge(START, "chatbot")

# Conditional logic: If the LLM wants a tool, go to "tools". Otherwise, end the graph.
graph_builder.add_conditional_edges(
    "chatbot", 
    tools_condition 
)
# After the tool runs, it must always go back to the chatbot to read the data
graph_builder.add_edge("tools", "chatbot")

# Compile into a runnable application
agent_graph = graph_builder.compile()

# 6. Execute the Graph
if __name__ == "__main__":
    query = "Who is cheaper: Bukayo Saka or Cole Palmer?"
    
    # We pass the System Message (persona) and Human Message (query) to start the state
    initial_state = {
        "messages": [
            SystemMessage(content="You are an expert Fantasy Premier League assistant."),
            HumanMessage(content=query)
        ]
    }
    
    print(f"User: {query}\n")
    print("Graph is running...\n")
    
    # Stream the output so we can see the graph moving between nodes
    for event in agent_graph.stream(initial_state):
        for value in event.values():
            if "messages" in value:
                message = value["messages"][-1]
                # Print the AI's final response or tool call requests
                if message.content:
                    print(f"Agent: {message.content}")